In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder

# Set display options
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print("All imports successful.")

# Load processed data
X_train = np.load('../models/X_train_processed.npy')
X_val = np.load('../models/X_val_processed.npy')
X_test = np.load('../models/X_test_processed.npy')
y_train_str = np.load('../models/y_train.npy', allow_pickle=True)
y_val_str = np.load('../models/y_val.npy', allow_pickle=True)
y_test_str = np.load('../models/y_test.npy', allow_pickle=True)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

# Encode string labels to integers for XGBoost
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_str)
y_val = label_encoder.transform(y_val_str)
y_test = label_encoder.transform(y_test_str)

# Store class names for later
class_names = label_encoder.classes_.tolist()
print(f"\nClass mapping: {dict(zip(range(len(class_names)), class_names))}")
print(f"y_train unique: {np.unique(y_train)}")

All imports successful.
X_train: (169, 100)
X_val:   (56, 100)
X_test:  (57, 100)

Class mapping: {0: 'Adventure', 1: 'Budget', 2: 'Culture', 3: 'Family', 4: 'Luxury', 5: 'Relaxation'}
y_train unique: [0 1 2 3 4 5]


In [6]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss'
    )
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
results = []

# For XGBoost, compute sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("=" * 70)
print("TRAINING AND EVALUATING MODELS")
print("=" * 70)

for name, model in models.items():
    print(f"\n{'─' * 70}")
    print(f"Model: {name}")
    print(f"{'─' * 70}")
    
    # For XGBoost, use sample_weight
    if name == 'XGBoost':
        model.fit(X_train, y_train, sample_weight=sample_weights)
        
        # Cross-validation with sample_weight (requires manual loop)
        cv_scores_acc = []
        cv_scores_f1 = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            X_tr_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
            y_tr_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
            sw_fold = sample_weights[train_idx]
            
            fold_model = XGBClassifier(
                n_estimators=100, max_depth=6, learning_rate=0.1,
                random_state=42, n_jobs=-1, eval_metric='mlogloss'
            )
            fold_model.fit(X_tr_fold, y_tr_fold, sample_weight=sw_fold)
            y_pred_fold = fold_model.predict(X_val_fold)
            cv_scores_acc.append(accuracy_score(y_val_fold, y_pred_fold))
            cv_scores_f1.append(f1_score(y_val_fold, y_pred_fold, average='macro'))
        
        cv_acc_mean = np.mean(cv_scores_acc)
        cv_acc_std = np.std(cv_scores_acc)
        cv_f1_mean = np.mean(cv_scores_f1)
        cv_f1_std = np.std(cv_scores_f1)
    else:
        # Standard cross-validation for LR and RF
        cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
        cv_acc_mean = cv_acc.mean()
        cv_acc_std = cv_acc.std()
        cv_f1_mean = cv_f1.mean()
        cv_f1_std = cv_f1.std()
        
        model.fit(X_train, y_train)
    
    # Validation set predictions
    y_val_pred = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    # Store results
    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': name,
        'params': json.dumps(model.get_params(), default=str),
        'cv_accuracy_mean': round(cv_acc_mean, 4),
        'cv_accuracy_std': round(cv_acc_std, 4),
        'cv_f1_macro_mean': round(cv_f1_mean, 4),
        'cv_f1_macro_std': round(cv_f1_std, 4),
        'val_accuracy': round(val_accuracy, 4),
        'val_f1_macro': round(val_f1, 4)
    })
    
    print(f"  CV Accuracy: {cv_acc_mean:.4f} ± {cv_acc_std:.4f}")
    print(f"  CV F1 (macro): {cv_f1_mean:.4f} ± {cv_f1_std:.4f}")
    print(f"  Val Accuracy: {val_accuracy:.4f}")
    print(f"  Val F1 (macro): {val_f1:.4f}")
    
    # Per-class metrics on validation set
    print(f"\n  Per-class metrics on validation set:")
    print(classification_report(y_val, y_val_pred, target_names=['Adventure', 'Budget', 'Culture', 'Family', 'Luxury', 'Relaxation']))

# Create results DataFrame
results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("SUMMARY - BEST MODEL BY VAL F1")
print("=" * 70)
print(results_df[['model_name', 'val_f1_macro', 'val_accuracy', 'cv_f1_macro_mean']].sort_values('val_f1_macro', ascending=False).to_string(index=False))

TRAINING AND EVALUATING MODELS

──────────────────────────────────────────────────────────────────────
Model: Logistic Regression
──────────────────────────────────────────────────────────────────────


c:\projects\smart-travel-planner\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  CV Accuracy: 0.9164 ± 0.0679
  CV F1 (macro): 0.9119 ± 0.0726
  Val Accuracy: 0.8214
  Val F1 (macro): 0.8153

  Per-class metrics on validation set:
              precision    recall  f1-score   support

   Adventure       0.75      0.67      0.71         9
      Budget       0.67      0.67      0.67         9
     Culture       0.75      0.60      0.67        10
      Family       0.82      1.00      0.90         9
      Luxury       1.00      1.00      1.00         9
  Relaxation       0.91      1.00      0.95        10

    accuracy                           0.82        56
   macro avg       0.82      0.82      0.82        56
weighted avg       0.82      0.82      0.82        56


──────────────────────────────────────────────────────────────────────
Model: Random Forest
──────────────────────────────────────────────────────────────────────
  CV Accuracy: 0.9046 ± 0.0676
  CV F1 (macro): 0.8988 ± 0.0723
  Val Accuracy: 0.8929
  Val F1 (macro): 0.8896

  Per-class metrics on valid

In [7]:
# Save results
import os
os.makedirs('../experiments', exist_ok=True)

results_df.to_csv('../experiments/results.csv', index=False)
print("Saved: backend/ml/experiments/results.csv")

print("\n" + "=" * 70)
print("NOTEBOOK 03 COMPLETE")
print("=" * 70)
print(f"Best model by Val F1: {results_df.loc[results_df['val_f1_macro'].idxmax(), 'model_name']}")
print(f"Best Val F1: {results_df['val_f1_macro'].max():.4f}")

Saved: backend/ml/experiments/results.csv

NOTEBOOK 03 COMPLETE
Best model by Val F1: Random Forest
Best Val F1: 0.8896
